In [6]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (348 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [17]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [20]:
!ollama --version

In [15]:
!ollama list

Error: could not connect to ollama server, run 'ollama serve' to start it


In [19]:
ollama_model_id = "gemma2:9b-instruct-q5_0"
# !pkill -f ollama

In [21]:
!nohup ollama serve &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
time=2026-11-17T23:28:23.296Z level=INFO source=routes.go:2586 msg="vram-based default context" total_vram="14.6 GiB" default_num_ctx=4096
time=2026-11-17T23:28:44.181Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_

In [22]:
!ollama pull {ollama_model_id}

In [23]:
!nohup ollama run {ollama_model_id}  &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
print_info: EOG token             = 107 '<end_of_turn>'
print_info: EOG token             = 213 '</s>'
print_info: max token length      = 93
load_tensors: loading model tensors, this can take a while... (load_mode = none)
⠏ ⠙ ⠹ ⠹ load_tensors: offloading output layer to GPU
load_tensors: offloading 41 repeating layers to GPU
load_tensors: offloaded 43/43 layers to GPU
load_tensors:        CUDA0 model buffer size =  6177.46 MiB
load_tensors:    CUDA_Host model buffer size =   717.77 MiB
⠼ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠴ ⠧ ⠧ ⠏ ⠋ ⠋ ⠹ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ 

In [27]:
%%bash
curl http://localhost:11434/api/chat -d '{
  "model": "gemma2:9b-instruct-q5_0",
  "stream": false,
  "messages": [
    { "role": "user", "content": "What were heros of titanic movie?" }
  ]
}'

{"model":"gemma2:9b-instruct-q5_0","created_at":"2026-11-17T23:33:07.457808162Z","message":{"role":"assistant","content":"It depends on what you mean by \"heroes\" in the context of the Titanic movie. \n\n**Traditional Heroes:**\n\n* **Jack Dawson:**  He's often seen as the classic hero, risking his life for Rose, displaying bravery and selflessness.\n\n* **Captain Edward John Smith:**  He stays on the ship until the very end, going down with it, which many see as an act of duty and heroism.\n\n**Unconventional Heroes:**\n\n* **Rose DeWitt Bukater:** While not a traditional hero, Rose shows courage and resilience in facing her own mortality and challenging societal expectations.\n* **Molly Brown:**  Known as \"The Unsinkable Molly Brown,\" she actively helps other passengers and even organizes efforts to keep lifeboats afloat.\n\n**It's important to note:**\n\n* The movie focuses on the human stories of survival and tragedy, and it doesn't necessarily present clear-cut \"heroes\" and \

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1671  100  1520  100   151    125     12  0:00:12  0:00:12 --:--:--   386


## **Ngrok**

In [28]:
from google.colab import userdata
ngrok_auth=userdata.get('colab-ngrok')

In [29]:
!pip install pyngrok==7.2.0

In [30]:
from google.colab import userdata
from pyngrok import ngrok, conf

ngrok_auth = userdata.get('colab-ngrok')

conf.get_default().auth_token = ngrok_auth

port = "11434"

public_url = ngrok.connect(port).public_url
print(public_url)

https://shortness-deceiving-flashily.ngrok-free.dev


In [31]:
%%bash
curl https://b762-34-168-215-24.ngrok-free.app/api/chat -d '{
  "model": "gemma2:9b-instruct-q5_0",
  "stream": false,
  "messages": [
    { "role": "user", "content": "ما عاصمة مصر؟" }
  ]
}'

<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2490  100  2348  100   142  52089   3150 --:--:-- --:--:-- --:--:-- 56590


---
------
-------


## After Updates

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
ollama_model_id = "gemma2:9b-instruct-q5_0"

In [3]:
!nohup bash -c "OLLAMA_HOST=0.0.0.0:8000 OLLAMA_ORIGIN=* ollama serve" &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
slot      release: id  0 | task 386 | stop processing: n_tokens = 267, truncated = 0
srv  update_slots: all slots are idle
[GIN] 2026/11/17 - 23:33:07 | 200 |   12.13s |       127.0.0.1 | POST     "/api/chat"
time=2026-11-17T23:53:33.993Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:8000 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAM

In [4]:
!nohup bash -c "OLLAMA_HOST=0.0.0.0:8000 OLLAMA_ORIGIN=* ollama run {ollama_model_id}" &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
print_info: max token length      = 93
load_tensors: loading model tensors, this can take a while... (load_mode = none)
⠋ ⠹ ⠸ ⠸ time=2026-11-17T23:56:12.525Z level=INFO source=llama_server.go:1406 msg="waiting for llama-server to become available" status="llm server not responding"
⠴ load_tensors: offloading output layer to GPU
load_tensors: offloading 41 repeating layers to GPU
load_tensors: offloaded 43/43 layers to GPU
load_tensors:        CUDA0 model buffer size =  6177.46 MiB
load_tensors:    CUDA_Host model buffer size =   717.77 MiB
⠴ ⠧ time=2026-11-17T23:56:12.776Z level=INFO source=llama_server.go:1406 msg="waiting for llama-server to become available" status="llm server loading model"
⠇ ⠏ ⠋ ⠙ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠏ ⠏ 

In [5]:
%%bash
curl http://localhost:8000/api/chat -d '{
  "model": "gemma2:9b-instruct-q5_0",
  "stream": false,
  "messages": [
    { "role": "user", "content": "What were heros of titanic movie?" }
  ]
}'

{"model":"gemma2:9b-instruct-q5_0","created_at":"2026-11-17T23:56:56.657423897Z","message":{"role":"assistant","content":"It's tricky to say definitively who the \"heroes\" of the Titanic movie are, as the film presents a complex and multifaceted story. \n\n**Some characters often seen as heroic include:**\n\n* **Jack Dawson:**  He's the quintessential romantic hero, willing to sacrifice everything for Rose and ultimately giving his life to save her.\n* **Rose DeWitt Bukater:**  While initially trapped by societal expectations, Rose ultimately chooses love and freedom over a life of privilege. Her courage and resilience in the face of disaster are inspiring.\n* **Molly Brown:**  This real-life figure is portrayed as a strong, outspoken woman who helps others and rallies passengers to stay calm during the sinking.\n\n**However, the film also explores heroism in less conventional ways:**\n\n* **The crew:**  The film portrays the crew members, particularly Captain Smith, as dedicated to t

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1945  100  1794  100   151    157     13  0:00:11  0:00:11 --:--:--   428


In [6]:
from google.colab import userdata
from pyngrok import ngrok, conf

ngrok_auth = userdata.get('colab-ngrok')

conf.get_default().auth_token = ngrok_auth

port = "8000"

public_url = ngrok.connect(port).public_url
print(public_url)

https://shortness-deceiving-flashily.ngrok-free.dev


In [7]:
%%bash
curl https://shortness-deceiving-flashily.ngrok-free.dev/api/chat -d '{
  "model": "gemma2:9b-instruct-q5_0",
  "stream": false,
  "messages": [
    { "role": "user", "content": "ما عاصمة مصر؟" }
  ]
}'

{"model":"gemma2:9b-instruct-q5_0","created_at":"2026-11-17T23:58:34.56575193Z","message":{"role":"assistant","content":"عاصمة مصر هي **القاهرة**. \n"},"done":true,"done_reason":"stop","total_duration":677882939,"load_duration":1571729,"prompt_eval_count":14,"prompt_eval_cached_count":0,"prompt_eval_duration":158695000,"eval_count":11,"eval_duration":398130000}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   522  100   380  100   142    274    102  0:00:01  0:00:01 --:--:--   377
